In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../../data")

pax = pd.read_csv(DATA_DIR / "AA_PAX_FORECASTING.csv")
airport = pd.read_csv(DATA_DIR / "AA_AIRPORT_INFO.csv")

df = pax
TARGET = "OD_PAX"

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df[TARGET].describe()

In [ ]:
df[TARGET].skew()

In [ ]:
plt.hist(df[TARGET].dropna(), bins=50)
plt.xlabel("OD_PAX")
plt.ylabel("Frequency")
plt.title("OD_PAX Distribution")
plt.show()

In [ ]:
plt.hist(np.log1p(df[TARGET].dropna()), bins=50)
plt.xlabel("log1p(OD_PAX)")
plt.ylabel("Frequency")
plt.title("Log-Transformed OD_PAX Distribution")
plt.show()

In [ ]:
plt.boxplot(df[TARGET].dropna(), vert=False)
plt.xlabel("OD_PAX")
plt.title("OD_PAX Boxplot")
plt.show()

OD_PAX is strongly right-skewed in its original form, while the log-transformed target is much more symmetric and approximately bell-shaped. Therefore, I recommend modeling log1p(OD_PAX) rather than the raw target. This should reduce the influence of the highest-demand routes while preserving those observations instead of deleting them as outliers.

In [ ]:
df.groupby(["ORG_AIRPRT", "DES_AIRPRT"])[TARGET].agg(
    ["count", "mean", "median", "max"]
).sort_values("max", ascending=False).head(20)

In [ ]:
q1 = df[TARGET].quantile(0.25)
q3 = df[TARGET].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outlier_mask = (df[TARGET] < lower) | (df[TARGET] > upper)

print("Potential outliers:", outlier_mask.sum())

The target contains high-end statistical outliers, but these should not be removed automatically. Passenger demand naturally varies substantially across routes, and high values may represent legitimate high-volume airport pairs rather than bad data. Therefore, extreme OD_PAX observations should be retained unless there is clear evidence that they are erroneous or corrupted.

Because OD_PAX is strongly right-skewed while log1p(OD_PAX) is much more symmetric, I recommend training the model on log1p(OD_PAX) using squared-error loss. This reduces the influence of extremely high-demand routes without removing valid observations. Predictions can later be converted back to the original passenger-count scale using expm1. Raw-target squared error can be used as a baseline, while MAE or Huber loss could be tested as more robust alternatives.

y_log = np.log1p(df[TARGET])

print("Raw skew:", df[TARGET].skew())
print("Log skew:", y_log.skew())

In [ ]:
df.isna().sum()

In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False)

The main passenger dataset contains no missing values in the observed columns. Therefore, no imputation is needed for the core passenger-modeling table. Missingness in the separate airport reference table should be handled independently based on what each airport metadata field represents.

OD_REV_USD is a likely leakage feature because realized revenue would only be known after passenger demand has occurred. Therefore, it should not be used to predict OD_PAX. The Unnamed: 0 column is an index artifact and should also be dropped. SF_LOAD_TMS should not be used until its meaning and availability at prediction time are confirmed.

In [ ]:
df.columns.tolist()

In [ ]:
df.select_dtypes(include=np.number).corr()[TARGET].sort_values(ascending=False)

In [ ]:
print("Origin airports:", df["ORG_AIRPRT"].nunique())
print("Destination airports:", df["DES_AIRPRT"].nunique())

The EDA points toward a gradient-boosted tree model such as XGBoost or LightGBM. The dataset contains high-cardinality airport categories, seasonal effects, and likely nonlinear relationships between route, time, and passenger demand. These characteristics are better suited to boosted trees than to a simple linear model.

| Column / feature | Decision | Reason |
|---|---|---|
| OD_PAX | target | Passenger demand is the quantity being predicted |
| log1p(OD_PAX) | engineer | Reduces right skew and limits the influence of extreme values |
| Extreme OD_PAX values | keep | High-demand routes can legitimately have large passenger counts |
| OD_REV_USD | leak / drop | Realized revenue would not be appropriate information at prediction time |
| Unnamed: 0 | drop / id | Index artifact with no predictive meaning |
| ORG_AIRPRT | keep | Origin airport is important route information |
| DES_AIRPRT | keep | Destination airport is important route information |
| DEP_MONTH | engineer | Captures seasonality |
| DEP_YEAR | keep | Captures changes over time |
| SF_LOAD_TMS | review / likely drop | Meaning and availability at prediction time should be confirmed before use |